In [ ]:
!pip install -U transformers

In [ ]:
!pip install -U transformers

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Load model and tokenizer
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# 2. Filtering mechanism (simple keyword check)
def is_python_question(prompt: str) -> bool:
    keywords = ["python", "code", "function", "class", "import", "def", "loop", "list", "dict"]
    prompt_lower = prompt.lower()
    return any(kw in prompt_lower for kw in keywords)

def answer_question(prompt: str):
    if not is_python_question(prompt):
        return "I can only answer Python coding questions."

    # Nudge GPT-2 into answering with Python code
    context_prompt = f"Q: {prompt}\nA (Python code only):\n```python\n"

    response = pipe(
        context_prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.encode("```")[0]  # stop when it hits ```
    )

    generated = response[0]["generated_text"]

    # Extract code after ```python
    if "```python" in generated:
        code = generated.split("```python")[-1].split("```")[0].strip()
        return f"```python\n{code}\n```"
    else:
        return generated


In [ ]:
print(answer_question("How do I write a for loop in Python?"))

In [ ]:
print(answer_question("What is the recipe of Ice Cream?"))